# သင်ခန်းစာ 10 - ထုတ်လုပ်မှုရှိ AI အေးဂျင့်များ

ဒီ သင်ခန်းစာမှာ သင်သည် Microsoft Agent Framework (Python) ကို အသုံးပြု၍ AI အေးဂျင့်များအတွက် **ထုတ်လုပ်မှု ပုံစံများ** ကို လေ့လာမယ်။ ကျွန်တော်တို့ ဖော်ပြမယ့် အချက်များမှာ:

- **Observability** — အေးဂျင့်အပြန်အလှန် လုပ်ဆောင်မှုများတွင် အချိန်တိုင်းတာခြင်းနှင့် မှတ်တမ်းတင်ခြင်း ထည့်သွင်းခြင်း
- **Evaluation** — တုံ့ပြန်မှု အရည်အသွေးကို အဆင့်သတ်မှတ်ရန် အကဲဖြတ် အေးဂျင့်ကို အသုံးပြုခြင်း
- **Cost management** — token အသုံးချမှုများကို အကောင်းဆုံးဖြစ်အောင် ပြုလုပ်ခြင်းနှင့် မော်ဒယ် ရွေးချယ်မှုဆိုင်ရာStrategies

ဤ ရာဇဝင်မှာ အသုံးပြုသူများအား ခရီးစဉ်စီစဉ်ရာတွင် ကူညီပေးသော **ခရီးသွား အေးဂျင့်** တစ်ခုကို ဥပမာထားပြီး၊ ထို့အပေါ်တွင် ကြီးကြပ်မှုနှင့် အကဲဖြတ်မှုများကို ထပ်တူတပ်ဆင်ထားသည်။


## ပြင်ဆင်ခြင်း


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ထုတ်လုပ်ရေးဆိုင်ရာ စဉ်းစားချက်များ

AI agent များကို prototype မှ production သို့ ရွှေ့ပြောင်းရန် သုံးခုသော အခြေခံအချက်များကို သေချာစွာ ဂရုစိုက်ရန် လိုအပ်သည်။

1. **ကြည့်ရှုနိုင်မှု** — agent က ဘာလုပ်နေပုံ၊ လုပ်ဆောင်ချိန် ကျော်ကြာမှု၊ နှင့် ဘာအရာများကို ခေါ်သုံးနေသည်ဆိုတာတို့ကို မြင်သာစေရန် လိုအပ်သည်။ tracing နှင့် logging မရှိပါက ထုတ်လုပ်ရေးပြဿနာများကို အမှားရှာဖွေ ဆန်းစစ်ဖြေရှင်းရခြင်းမှာ ပျမ်းမျှအားဖြင့် မဖြစ်နိုင်ပါ။

2. **အကဲဖြတ်ခြင်း** — အလိုအလျောက် အရည်အသွေး စစ်ဆေးမှုများက agent ၏ တုံ့ပြန်ချက်များကို အချိန်လာလျှင်မှန်ကန်၊ ပြည့်စုံနှင့် အထောက်အကူဖြစ်နေသေးကြောင်း အာမခံပေးသည်။ အကဲဖြတ်ရေး agent တစ်ခုက သတ်မှတ်ထားသော စံနှုန်းများအပေါ် မူတည်၍ တုံ့ပြန်ချက်များကို အမှတ်ပေးနိုင်သည်။

3. **ကုန်ကျစရိတ် စီမံခန့်ခွဲမှု** — Token အသုံးပြုမှုသည် ကုန်ကျစရိတ်ကို တိုက်ရိုက် ထိခိုက်စေသည်။ prompt optimization, မော်ဒယ်ရွေးချယ်ခြင်း နှင့် caching ကဲ့သို့သော မဟာဗျူဟာများက အရည်အသွေးကို မလျော့စေဘဲ စရိတ်ကို ထိန်းချုပ်ရာတွင် ကူညီပေးသည်။


## ကြည့်ရှုနိုင်သော အေဂျင့် တည်ဆောက်ခြင်း

ကျွန်ုပ်တို့သည် ခရီးသွားဆိုင်ရာ ကိရိယာများကို သတ်မှတ်ပြီး အေဂျင့် ခေါ်ယူမှုအား အချိန်တိုင်းတာစနစ်ဖြင့် ဖုံးအုပ်ထားကာ နောက်ကျမှုကို စောင့်ကြည့်နိုင်အောင် ပြုလုပ်သည်။ ထုတ်လုပ်မှုတွင် OpenTelemetry သို့မဟုတ် ဆင်တူသော tracing backend တစ်ခုနှင့် ပေါင်းစည်းရမည်။


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## သုံးသပ်မှု ပုံစံများ

ထုတ်လုပ်မှုတွင် အများအားဖြင့် အသုံးပြုသော ပုံစံတစ်ခုမှာ ဒုတိယ အေးဂျင့်ကို **စိစစ်သူ** အဖြစ် အသုံးပြုခြင်းဖြစ်သည်။ စိစစ်သူသည် သတ်မှတ်ထားသော စံသတ်မှတ်ချက်များဖြစ်သည့် ပြည့်စုံမှု၊ တိကျမှု နှင့် အသုံးဝင်မှု စသည်တို့အပေါ် အဓိက အေးဂျင့်၏ တုံ့ပြန်ချက်ကို အမှတ်ပေးသည်။

ဤသည်က အောက်ပါအရာများကို ခွင့်ပြုသည်:
- အသုံးပြုသူထံ ရောက်ရန် မတိုင်ခင် အလိုအလျောက် အရည်အသွေး စစ်ဆေးမှုများ
- prompts များ သို့မဟုတ် မော်ဒယ်များ ပြောင်းလဲသောအခါ နောက်ပြန်ကျမှုများကို တွေ့ရှိနိုင်ခြင်း
- အချိန်အလိုက် အေးဂျင့် စွမ်းဆောင်ရည်ကို ဆက်တိုက် စောင့်ကြည့်ခြင်း


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ကုန်ကျစရိတ် စီမံခန့်ခွဲမှု နည်းဗျူဟာများ

ထုတ်လုပ်မှု AI agent များအတွက် ကုန်ကျစရိတ် ထိန်းချုပ်ခြင်းသည် အလွန်အရေးကြီးသည်။ အဓိက နည်းဗျူဟာများမှာ အောက်ပါအတိုင်း ဖြစ်ပါသည်။

| နည်းဗျူဟာ | ဖော်ပြချက် |
|---|---|
| **Prompt အကောင်းဆုံးပြင်ဆင်ခြင်း** | စနစ်ညွှန်ကြားချက်များကို တိုတိုခလုတ်ရှင်းလင်းစွာ ထားပါ။ ထပ်တူထပ်မွှားသော အကြောင်းအရာများကို ဖယ်ရှား၍ ထည့်သွင်း token များကို လျော့ချပါ။ |
| **မော်ဒယ် ရွေးချယ်ခြင်း** | အလွယ်တကူ ပြုလုပ်နိုင်သော လုပ်ငန်းများ (ဥပမာ အမျိုးအစားသတ်မှတ်ခြင်း သို့မဟုတ် ထုတ်ယူခြင်း) အတွက် သေးငယ်ပြီး စျေးသက်သာသော မော်ဒယ်များ (e.g. GPT-4o-mini) ကို အသုံးပြုပြီး၊ ရှုပ်ထွေးသော သုံးသပ်ချက်များအတွက် ကြီးမားသည့် မော်ဒယ်များကို သီးသန့်ထားပါ။ |
| **Caching** | ကိရိယာရလဒ်များနှင့် အမြဲမေးစေသော မေးခွန်းများကို cache ထား၍ ထပ်မံဖြစ်ပေါ်မည့် API ခေါ်ဆိုမှုများကို ရှောင်ကြဉ်ပါ။ |
| **Token အကန့်အသတ်များ** | `max_tokens` အကန့်အသတ်များကို သတ်မှတ်၍ မျှော်လင့်မထားသော အရှည်များ ထွက်ပေါ်ခြင်းကို တားဆီးပါ။ |
| **အစုလိုက်ဆောင်ရွက်ခြင်း (Batching)** | ဖြစ်နိုင်သမျှ များစွာသော အသုံးပြုသူ မေးခွန်းများကို တစ်ခေါက် API ခေါ်ဆိုမှုတစ်ခုအဖြစ် အစုလိုက်ထားပါ။ |

လက်တွေ့တွင် အဆင့်ခွဲထားသည့် နည်းလမ်းသည် အကျိုးရှိသည် — ရိုးရှင်းပြီး တောင်းဆိုချက်များကို မြန်နှုန်းမြင့်၊ စျေးသက်သာသော မော်ဒယ်သို့ ပို့ပြီး ရှုပ်ထွေးသော မေးခွန်းများကိုသာ ပိုမိုစွမ်းဆောင်နိုင်သော မော်ဒယ်သို့ မြှင့်တင်ပါ။


## အကျဉ်းချုပ်

ဒီသင်ခန်းစာတွင် သင် သင်ယူခဲ့သည့်အချက်များမှာ:

1. **ကြည့်မြင်နိုင်မှု ထည့်သွင်းခြင်း** အချိန်တိုင်းတာခြင်းနှင့် မှတ်တမ်းတင်ခြင်းများဖြင့် အေးဂျင့်များ၏ ဆက်သွယ်မှုများတွင် tracing နှင့် monitoring အတွက် အခြေခံအလုပ်များကို တည်ဆောက်ခြင်း။
2. **အေးဂျင့်၏ ဖြေကြားချက်များကို အကဲဖြတ်ခြင်း** ကို evaluator agent တစ်ခု အသုံးပြု၍ ပြည့်စုံမှု၊ တိကျမှုနှင့် အထောက်အကူဖြစ်မှုတို့အား အမှတ်ပေးကာ အလိုအလျောက် စစ်ဆေးခြင်း။
3. **ကုန်ကျစရိတ် စီမံခန့်ခွဲခြင်း** ကို prompt အ tốiင်မိုက်ဇေးရှင်း၊ မော်ဒယ်ရွေးချယ်မှု၊ caching နှင့် token ဘတ်ဂျက်များမှတဆင့် အကောင်အထည်ဖော်ခြင်း။

ဤ ထုတ်လုပ်မှု နည်းပုံများသည် သင့် AI အေးဂျင့်များကို ယုံကြည်စိတ်ချရ၊ တိုင်းတာနိုင်ပြီး စျေးနှုန်းထိရောက်စွာ လည်ပတ်နိုင်စေရန် ကူညီပေးသည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
တာဝန်မရှိကြောင်း ကြေညာချက်:

ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှုဖြစ်သော [Co-op Translator](https://github.com/Azure/co-op-translator) ဖြင့် ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်တင်အလိုအလျောက် ဘာသာပြန်ချက်များတွင် အမှားများ သို့မဟုတ် မတိကျမှုများ ပါရှိနိုင်ကြောင်း သတိပေးပါသည်။ မူလစာတမ်းကို မူလဘာသာဖြင့်သာ တရားဝင် အရင်းအမြစ်အဖြစ် သတ်မှတ်သင့်ပါသည်။ အရေးပါသော အချက်အလက်များအတွက် သာမာန်အဆင့်မဟုတ်၍ လူ့ပရော်ဖက်ရှင်နယ် ဘာသာပြန်ခြင်းကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းကြောင့် ဖြစ်တတ်သော နားမလည်မှုများ သို့မဟုတ် မမှန်ကန်စွာ ဖတ်ယူခြင်းများအတွက် ကျွန်ုပ်တို့ အာမခံမှု၊ တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
